# Praktikum: Face Detection dan Klasifikasi Wajah

## A. Mengakses Dataset Face Detection

Dataset sudah di-crop (hanya wajah) dan di-resize, terletak di Google Drive: `/content/drive/MyDrive/Dataset_Face_Only`.
Asumsikan struktur folder: subfolder per kelas (misal: senang, sedih, marah, dll.).

In [ ]:
# Mount Google Drive
from google.colab import drive
drive.mount('/content/drive')

# Path dataset
dataset_path = '/content/drive/MyDrive/Dataset_Face_Only'
print('Dataset accessed at:', dataset_path)

## B. Preprocessing Dataset untuk Project Face Detection

Preprocessing: split train/val, augmentation, normalisasi.

In [ ]:
import tensorflow as tf
from tensorflow.keras.preprocessing.image import ImageDataGenerator
from sklearn.model_selection import train_test_split
import os

# ImageDataGenerator untuk preprocessing dan augmentation
train_datagen = ImageDataGenerator(
    rescale=1./255,
    rotation_range=20,
    width_shift_range=0.2,
    height_shift_range=0.2,
    horizontal_flip=True,
    validation_split=0.2  # 80% train, 20% val
)

# Load data
train_generator = train_datagen.flow_from_directory(
    dataset_path,
    target_size=(224, 224),
    batch_size=32,
    class_mode='categorical',
    subset='training'
)

val_generator = train_datagen.flow_from_directory(
    dataset_path,
    target_size=(224, 224),
    batch_size=32,
    class_mode='categorical',
    subset='validation'
)

print('Preprocessing selesai. Jumlah kelas:', train_generator.num_classes)

## C. Alur Proses (Mermaid Diagram)

```mermaid
graph TD
    A[Mengakses Dataset dari Google Drive] --> B[Preprocessing: Resize, Augmentation, Split Train/Val, Normalisasi]
    B --> C[Membuat Model CNN Klasifikasi Wajah]
    C --> D[Training Model dengan Data Train]
    D --> E[Evaluasi pada Data Validation]
    E --> F[Prediksi pada Gambar Baru]
```

## D. Membuat Model Klasifikasi Wajah

Menggunakan CNN sederhana dengan Keras/TensorFlow.

In [ ]:
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Conv2D, MaxPooling2D, Flatten, Dense, Dropout

# Buat model CNN
model = Sequential([
    Conv2D(32, (3,3), activation='relu', input_shape=(224, 224, 3)),
    MaxPooling2D(2,2),
    Conv2D(64, (3,3), activation='relu'),
    MaxPooling2D(2,2),
    Conv2D(128, (3,3), activation='relu'),
    MaxPooling2D(2,2),
    Flatten(),
    Dense(512, activation='relu'),
    Dropout(0.5),
    Dense(train_generator.num_classes, activation='softmax')  # Sesuaikan num_classes
])

model.compile(optimizer='adam',
              loss='categorical_crossentropy',
              metrics=['accuracy'])

model.summary()

In [ ]:
# Training model
history = model.fit(
    train_generator,
    epochs=20,
    validation_data=val_generator
)

# Simpan model
model.save('/content/drive/MyDrive/face_classification_model.h5')
print('Model berhasil dibuat dan disimpan!')

In [ ]:
# Plot hasil training
import matplotlib.pyplot as plt

plt.plot(history.history['accuracy'], label='Train Accuracy')
plt.plot(history.history['val_accuracy'], label='Val Accuracy')
plt.legend()
plt.show()